# EEG-Twin — run the pipeline

CEBRA trajectory embedding → Transformer digital twin → paper figures.

**Runtime → Change runtime type → GPU** before running anything.

This notebook lives in the same Drive folder as the data. It reads the data
from there, clones the code from GitHub, and writes results back beside the
data. Run the cells top to bottom.

In [ ]:
#@title Settings { display-mode: "form" }

#@markdown Leave blank to use the folder this notebook is in.
DATA_FOLDER = ""  #@param {type:"string"}

#@markdown Twin training: 10 seeds is the paper setting. Lower it if the
#@markdown Colab session times out before it finishes.
TWIN_SEEDS = 10  #@param {type:"slider", min:1, max:10, step:1}

#@markdown A 4-patient, 1-epoch pass that checks the wiring in ~10 min.
#@markdown The numbers it produces are meaningless.
SMOKE_TEST = False  #@param {type:"boolean"}

REPO = "https://github.com/ahmostafa147/Cardiac_Arrest_Coma_Recovery_Twin_Prediction.git"

## 1 · Mount Drive and find the data

In [ ]:
from google.colab import drive
from pathlib import Path
import os, glob

drive.mount('/content/drive')

if DATA_FOLDER:
    SRC = Path('/content/drive/MyDrive') / DATA_FOLDER
else:
    # the folder holding this notebook
    hits = glob.glob('/content/drive/MyDrive/**/PPNet_data_train.npz', recursive=True)
    if not hits:
        raise SystemExit('Could not find PPNet_data_train.npz anywhere in My Drive.\n'
                         'Set DATA_FOLDER above to the folder you uploaded.')
    SRC = Path(hits[0]).parent

need = ['PPNet_data_train.npz', 'PPNet_data_test.npz', 'ICARE_clinical.csv']
gone = [n for n in need if not (SRC / n).exists()]
if gone:
    raise SystemExit(f'{SRC} is missing: {gone}')

print(f'data: {SRC}\n')
for p in sorted(SRC.iterdir()):
    if p.is_file():
        print(f'   {p.stat().st_size/1e9:7.2f} GB  {p.name}')

## 2 · Install

Run this, **restart the session** when it says to, then run it again.

In [ ]:
import subprocess, sys, importlib.util
if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', '-q', REPO, '/content/repo'], check=True)
%cd /content/repo
!git pull -q 2>/dev/null
!pip install -q -r requirements.txt 2>&1 | grep -viE "dependency resolver|which is incompatible|^$" | head -3

if all(importlib.util.find_spec(m) for m in ('torch', 'cebra', 'plotly')):
    print('\nready')
else:
    print('\ninstalled — RESTART THE SESSION (Runtime > Restart session), then re-run this cell')

## 3 · Stage the data into the repo

In [ ]:
import shutil
%cd /content/repo
Path('data/dataset').mkdir(parents=True, exist_ok=True)
Path('data/tables').mkdir(parents=True, exist_ok=True)

for n in ('PPNet_data_train.npz', 'PPNet_data_test.npz'):
    if not Path('data/dataset', n).exists():
        print('copying', n, '...', flush=True)
        shutil.copy(SRC / n, 'data/dataset/')
for n in ('ICARE_clinical.csv', 'split_train.csv', 'split_test.csv'):
    if (SRC / n).exists():
        shutil.copy(SRC / n, 'data/tables/')

import numpy as np
for s in ('train', 'test'):
    d = np.load(f'data/dataset/PPNet_data_{s}.npz', allow_pickle=True)
    print(f'  {s}: {len(np.unique(d["patient_ids"]))} patients, {len(d["patient_ids"]):,} segments')

## 4 · Run

Every stage in order, skipping whatever is already built and rebuilding
anything whose inputs changed. Safe to re-run after a disconnect — it picks up
where it stopped.

CEBRA is ~5 min on a T4. Twin training is the long pole and shows a per-seed
progress bar, so you can tell early whether it will finish.

In [ ]:
# Env goes on os.environ directly: the ! magic streams output to the notebook,
# subprocess.run() does not (its stdout bypasses Jupyter's display machinery).
if SMOKE_TEST:
    os.environ['CEBRA_SMOKE'] = os.environ['CEBRA_TWIN_SMOKE'] = '1'
    print('SMOKE TEST — 4 patients, 1 epoch. The numbers are meaningless.\n')
elif TWIN_SEEDS != 10:
    Path('sitecustomize.py').write_text(
        "import sys; sys.path.insert(0, 'src')\n"
        f"import config; config.TWIN_TRAIN.update(N_REF={TWIN_SEEDS}, "
        f"N_TWIN={TWIN_SEEDS}, N_ABL={TWIN_SEEDS}, N_CAL={TWIN_SEEDS})\n")
    os.environ['PYTHONPATH'] = '/content/repo'
    print(f'twin seeds set to {TWIN_SEEDS}\n')

!python -u run_all.py


## 5 · Save results back to the Drive folder

In [ ]:
DST = SRC / 'results'
DST.mkdir(exist_ok=True)

for sub in ('outputs', 'data/cebra', 'data/twin/handoffs'):
    s = Path(sub)
    if s.exists() and any(s.rglob('*')):
        d = DST / s.name
        shutil.rmtree(d, ignore_errors=True)
        shutil.copytree(s, d)
        print('saved', sub)

print(f'\n{DST}')
for p in sorted(DST.rglob('*')):
    if p.is_file():
        print(f'   {p.stat().st_size/1e6:8.1f} MB  {p.relative_to(DST)}')

## Results

`results/outputs/figures/` — each figure as interactive HTML, 600-dpi PNG and
vector PDF.

| | |
|---|---|
| `cebra/` | trajectory globes · twin in CEBRA space · prototype map |
| `twin/` | fig1-7 |
| `eval/` | confusion · CPC · centroid distances |

Open an HTML to rotate a globe — it prints the camera as a dict you can paste
into the script's `CAMERA` to lock that view for the static export.

## If it stops

**Session died.** Re-run cells 1-4. Finished stages are cached; you lose only
the stage that was running.

**Twin never finishes.** Drop `TWIN_SEEDS` to 2-3 and re-run cell 4. Figures
still build — the ensemble is just smaller.

**Something looks stale.** The runner rebuilds a stage when its inputs are
newer than its outputs. To force everything: `!python run_all.py --force`.